# Exercise 2
## 1 Data Set, 1 Model, and 6 Training Algorithms

# Utilities
Shared code used by all problems: imports, data handling, the Weibull model, and helpers to record and plot fits.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt
import scipy.special as spec
import scipy.integrate as integ

In [ ]:
def load_myspace_data():
    df = pd.read_csv('dataMyspace201303.csv')

    second = np.array(df.iloc[:, 1])

    searchCount = np.trim_zeros(second, 'f')

    time = np.array(np.linspace(1, len(searchCount), len(searchCount))).T

    return time, searchCount

def to_samples(t, c):
    # histogram -> individual observations (non-elegant way)
    return np.repeat(t, c)

def to_distribution(c):
    return (c/np.sum(c)) * 0.98

In [ ]:
def weibull_pdf(t, a, b):
    return (a/b) * (t/b)**(a-1) * np.exp(-((t/b)**a))

def weibull_cdf(t, a, b):
    return 1 - np.exp(-((t/b)**a))

In [ ]:
results = {}

def record(name, a, b):
    results[name] = {'alpha': a, 'beta': b}
    print(f'{name:<28} alpha = {a:8.4f}   beta = {b:8.3f}')

def plot_fit(name, ax=None):
    a, b = results[name]['alpha'], results[name]['beta']
    ax = ax or plt.figure().gca()
    ax.plot(t, h, color='lightgray', lw=2, label='data (normalized)')
    ax.plot(t, weibull_pdf(t, a, b), label='Weibull fit')
    ax.set_title(f'{name}\nalpha = {a:.3f}, beta = {b:.2f}')
    ax.set_xlabel('week after onset')
    ax.set_ylabel('probability')
    ax.legend()
    return ax

def run(name, fit_fn, show=True):
    a, b = fit_fn(t, c)
    record(name, a, b)
    if show:
        plot_fit(name)
        plt.show()

In [ ]:
t, c = load_myspace_data()
h = c / np.sum(c)              # normalized histogram, common scale for all comparisons

# Problem 2.1: setting the stage

## Task 2.1.1

In [ ]:
def plot_myspace_data(x, y):
    plt.figure()
    plt.plot(x, y, label='datapoints')

    plt.xlabel('week after onset')
    plt.ylabel('normalized search count')
    plt.legend()
    plt.show()

plot_myspace_data(t, c)

## Task 2.1.2
`weibull_pdf` / `weibull_cdf` are defined in the utilities section.

In [ ]:
ts = np.linspace(0, 10, 100)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for alpha in [0.5, 1.0, 1.5, 2.0, 2.5]:
    ax1.plot(ts, weibull_pdf(ts, alpha, 2.0), '-', label=f'alpha = {alpha}')
ax1.set_title('Weibull PDF; beta = 2.0')

for beta in [0.5, 1, 2, 3, 4]:
    ax2.plot(ts, weibull_pdf(ts, 2.5, beta), '-', label=f'beta = {beta}')
ax2.set_title('Weibull PDF; alpha = 2.5')

for ax in (ax1, ax2):
    ax.set_xlabel('t')
    ax.set_ylabel('f(t)')
    ax.legend()
plt.show()

# Problem 2.2: fitting a Weibull to a histogram (part 1)
**Method:** maximize the log-likelihood of the samples $D$ with Newton's method.

## Task 2.2.1

In [ ]:
def loss(a,b,D):
    N = len(D)
    return N * (np.log(a) - a*np.log(b)) + (a-1) * np.sum(np.log(D)) - np.sum((D/b)**a)

def dLa(a,b,D):
    N = len(D)
    return N/a - N*np.log(b) + np.sum(np.log(D)) - np.sum((D/b)**a * np.log(D/b))

def dLb(a,b,D):
    N = len(D)
    return a/b * (np.sum((D/b)**a)-N)

def dLa2(a,b,D):
    N = len(D)
    return -(N/(a*a)) - np.sum((D/b)**a * (np.log(D/b)**2))

def dLb2(a,b,D):
    N = len(D)
    return (a/(b*b)) * (N-(a+1)*np.sum((D/b)**a))

def dLab(a,b,D):
    N = len(D)
    return (1/b) * np.sum((D/b)**a) + (a/b) * np.sum((D/b)**a * np.log(D/b)) - N/b

In [ ]:
def newton_increment(a,b,D):
    dla  = dLa(a,b,D)
    dlb  = dLb(a,b,D)
    dla2 = dLa2(a,b,D)
    dlb2 = dLb2(a,b,D)
    dlab = dLab(a,b,D)

    mat1 = np.array([a,b]).T
    mat2 = np.linalg.inv(np.array([[dla2, dlab],[dlab, dlb2]]))
    mat3 = np.array([dla, dlb]).T

    return mat1 - mat2 @ mat3

def newton_update(D, iterations):
    a = 1
    b = 1

    for i in range(iterations):
        result = newton_increment(a,b,D)
        a = result[0]
        b = result[1]

    return a,b

def fit_newton(t, c, iterations=20):
    return newton_update(to_samples(t, c), iterations)

## Task 2.2.2

In [ ]:
run('2.2 Newton (ML)', fit_newton)

# Problem 2.3: fitting a Weibull to a histogram (part 2)
**Method:** least-squares fit of the scaled Weibull $A \cdot f(t \mid \alpha, \beta)$ to the counts with `scipy.optimize.curve_fit`.

## Task 2.3.1

In [ ]:
def f_tilde(x,A,a,b):
    return A * weibull_pdf(x,a,b)

def optimize_params_scipy(fun, t,c, A,a,b):
    popt, _ = opt.curve_fit(fun, t, c, p0=[A,a,b], bounds=([0,0,0],[np.inf, np.inf, np.inf]))
    A_opt, a_opt, b_opt = popt
    return A_opt, a_opt, b_opt

def fit_curve_fit(t, c):
    A, a, b = optimize_params_scipy(f_tilde, t,c, 1000,2,200)
    return a, b

## Task 2.3.2

In [ ]:
run('2.3 curve_fit (LSQ)', fit_curve_fit)

# Problem 2.4: fitting a Weibull to a histogram (part 3)
**Method:** weighted least squares on the bin probabilities $p(t) = F(t) - F(t-1)$ with `scipy.optimize.least_squares`.

## Task 2.4.1

In [ ]:
def p(t, a,b):
    # F(t) - F(t-1)
    return weibull_cdf(t, a,b) - weibull_cdf(t-1, a,b)

def residual(theta,t,c):
    a,b = theta
    w = 1/p(t,a,b)
    return np.sqrt(w) * (c - np.sum(c)*p(t, a,b))

def fit_weighted_lsq(t, c):
    res = opt.least_squares(residual, x0=[2,200], args=(t,c), bounds=([0,0], [np.inf,np.inf]))
    return res.x

## Task 2.4.2

In [ ]:
run('2.4 weighted least squares', fit_weighted_lsq)

# Problem 2.5: fitting a Weibull to a distribution (part 1)
**Method:** minimize the KL divergence between the Weibull and the data distribution $q$ with `scipy.optimize.minimize`.

## Task 2.5.1

In [ ]:
def Dkl_weibull(theta, t,q,):
    a,b = theta
    f = weibull_pdf(t,a,b)
    return np.sum(f*np.log(f/q))

def fit_kl(t, c):
    q = to_distribution(c)
    res = opt.minimize(Dkl_weibull, x0=[2,200], args=(t,q), bounds=[(0, np.inf), (0, np.inf)])
    return res.x

## Task 2.5.2

In [ ]:
run('2.5 KL divergence', fit_kl)

# Problem 2.6: fitting a Weibull to a distribution (part 2)
**Method:** method of moments, matching the mean and variance of $q$ (root finding for $\alpha$, closed form for $\beta$).

## Task 2.6.1

In [ ]:
def get_alpha(a,m,sq):
    part1 = spec.gamma(1+2/a)
    part2 = spec.gamma(1+1/a)**2

    return ((part1-part2)/part2) - sq/m**2

def fit_moments(t, c):
    q = to_distribution(c)
    m = np.sum(q*t)
    sq = np.sum(q* t**2) - m**2

    result = opt.root_scalar(get_alpha, args=(m,sq), bracket=[2,10])
    a = result.root
    b = m / spec.gamma(1+(1/a))
    return a, b

## Task 2.6.2

In [ ]:
run('2.6 method of moments', fit_moments)

# Problem 2.7: solving differential equations
**Method:** solve $F'(t) = \frac{\alpha}{\beta}\left(\frac{t}{\beta}\right)^{\alpha-1}(1 - F)$ numerically and compare with the analytical CDF.

## Task 2.7.1

In [ ]:
def dF(t, F, a,b):
    return a/b * (t/b)**(a-1) * (1-F)

def solve_cdf(t, a, b):
    result = integ.solve_ivp(fun=dF, t_span=(t[0], t[-1]), y0=[0], args=(a,b))
    return weibull_cdf(t, a, b), result.t, result.y[0]   # F_ana, t_num, F_num

a = 2.86
b = 215.91

F_ana, t_num, F_num = solve_cdf(t, a, b)

q = to_distribution(c)
Q_cum = np.cumsum(q)

## Task 2.7.2

In [ ]:
plt.figure()
plt.plot(t, q, label='datapoints')
plt.plot(t, Q_cum, label ='cumulative distribution Q')
plt.plot(t, F_ana, label ='analytical F_ana')
plt.plot(t_num, F_num, label ='numerical F_num')

plt.xlabel('week after onset')
plt.ylabel('normalized search count')
plt.legend()
plt.show()

# Comparison of all methods

In [ ]:
summary = pd.DataFrame(results).T
summary['RMSE vs data'] = [np.sqrt(np.mean((h - weibull_pdf(t, r.alpha, r.beta))**2))
                           for r in summary.itertuples()]
summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
for ax, name in zip(axes.flat, results):
    plot_fit(name, ax)

ax = axes.flat[-1]                         # 6th panel: all fits overlaid
ax.plot(t, h, color='lightgray', lw=3, label='data')
for name, r in results.items():
    ax.plot(t, weibull_pdf(t, r['alpha'], r['beta']), label=name)
ax.set_title('all fits')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()